In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DataType, TimestampType, FloatType
from pyspark.sql import Row

In [0]:
catalog_name = "ecommerce"

#Products

In [0]:
df_products  = spark.table(f"{catalog_name}.silver.slv_products")
df_brands = spark.table(f"{catalog_name}.silver.slv_brands")
df_category = spark.table (f"{catalog_name}.silver.slv_category")

###Create three temporary view

In [0]:
df_products.createOrReplaceTempView("v_products")
df_brands.createOrReplaceTempView("v_brands")
df_category.createOrReplaceTempView("v_category")

In [0]:
display(spark.sql("select * from v_products limit 5"))

In [0]:
#Making Sure we're on the right catalog
spark.sql(f"use catalog {catalog_name}")

In [0]:

%sql
--To Build brand x category mapping and trasform to Gold table
--create a CTE (Common Table Expression)

Create Or Replace Table gold.dim_products As

With brands_categories AS (

    SELECT
     b.brand_name,
     b.brand_code,
     c.category_name,
     c.category_code
     FROM v_brands b
     Inner join v_category c
     on 
     b.category_code = c.category_code
)


SELECT
    p.product_id,
    p.sku,
    p.category_code,
    COAlESCE(bc.category_name, 'Not Available') As category_name,
    p.brand_code,
    COALESCE(bc.brand_name, 'Not Available') As brand_name,
    p.color,
    p.size,
    p.material,
    p.weight_grams,
    p.length_cm,
    p.width_cm,
    p.height_cm,
    p.rating_count,
    p.file_name,
    p.ingest_timestamp
from v_products p
left join brands_categories bc
on
p.brand_code = bc.brand_code;


In [0]:
# Myanmar States

myanmar_region = {
    "MH":"West", "GJ":"West", "RJ" :"West",
    "KA":"South", "TN":"South", "TS":"South", "AP":"South", "KL":"South", "UP":"North", "WB": "North", "DL": "North"
}

#Austrila States
australia_region = {
    "VIC":"SouthEast", "WA":"West", "NSW": "East", "QLD": "NorthEast"
}

#United Kingdoms States
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR":"Northern Ireland", "SCT":"Scotland"
}

#United State States
us_region = {
    "MA":"NorthEast", "FL": "South", "NJ":"NorthEast", "CA":"West",
    "NY":"NorthEast", "TX":"South"

}

#UAE States
uae_region ={
    "AUH": "Abu Dhabi", "DU":"Dubai", "SHJ":"Sharjah"
}

#Singapore States

singapore_region ={
    "SG" : "Singapore"
}

#Canada States
canada_region = {
    "BC":"WEST", "AB":"WEST", "ON":"East", "QC":"East", "NS":"East", "IL":"Other"
}

#Combine into a master dictionary
country_state_map ={
    "Myanmar" : myanmar_region,
    "Australia" : australia_region,
    "United Kingdom" : uk_region,
    "United States" : us_region,
    "United Arab Emirates" : uae_region,
    "Singapore":singapore_region,
    "Canada" : canada_region
}

### Showing Dictionaries

In [0]:
country_state_map

In [0]:
# 1 flatten country_state_map into a list of Rows

rows =[]
for country, states in country_state_map.items():
    for state_code, region in states.items():
        rows.append(Row(country=country, state=state_code, region=region))
rows[:10]

###Creating Mapping DataFrame

In [0]:
df_region_mapping  = spark.createDataFrame(rows)

###Show Mapping

In [0]:
df_region_mapping.show(truncate = False)

###Reading Silver Customer Table - slv_customers

In [0]:
df_silver = spark.table(f'{catalog_name}.silver.slv_cusotmers')
display(df_silver.limit(5))

###Joining Region Mapping

In [0]:
df_gold = df_silver.join(df_region_mapping, on=['country', 'state'], how= 'left')
df_gold= df_gold.fillna({'region':'Other'})
display(df_gold.limit(5))

###Creating dim_customers table in Gold Layer

In [0]:
#Write raw data into the gold layer
df_gold.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
            .saveAsTable(f'{catalog_name}.gold.dim_customers')

#Date and Calendar

In [0]:
df_silver = spark.table(f'{catalog_name}.silver.slv_calendar')
display(df_silver.limit(5))

###Adding Colmnns  that we need 

In [0]:
df_gold = df_silver.withColumn("date_id",F.date_format(F.col("date"),"yyyyMMdd").cast("int"))

#Add a moth name column. Fo example ("January","February", etc)

df_gold = df_gold.withColumn("month_name",F.date_format(F.col("date"),"MMMM"))

#Add is_weekend column
df_gold = df_gold.withColumn(
    "is_weekend",
    F.when(F.col("day_name").isin("Saturday","Sunday"),1)\
        .otherwise(0)
)

display(df_gold)

In [0]:
desired_columns_order = ["date_id", "date", "year", "month_name", "day_name", "is_weekend", "quarter", "week", "ingested_at", "source_file"]

df_gold =df_gold.select(desired_columns_order)

display(df_gold)

#Create dim_date at gold layer

In [0]:
df_gold.write.format("delta")\
    .mode("overwrite")\
        .option("mergerSchema","true")\
            .saveAsTable(f"{catalog_name}.gold.dim_date")